# MRMS Module Demo

This notebook exercises the `mrms` module for downloading and processing NOAA MRMS (Multi-Radar/Multi-Sensor System) data from NOAA's public AWS S3 bucket. All results are returned as xarray Datasets in EPSG:4326.

**Supported products:**
- `QPE_RADAR_ONLY` (`RadarOnly_QPE_01H_00.00`) — 1-hour radar-only Quantitative Precipitation Estimation
- `COMPOSITE_REFLECTIVITY` (`CREF_1HR_MAX_00.50`) — 1-hour maximum composite reflectivity at 0.50° tilt

**Supported domains:** `CONUS`, `ALASKA`, `HAWAII`

**Public functions:**
- `download_mrms()` — download and decompress a single GRIB2 file
- `process_mrms()` — open a local GRIB2 file, apply a bounding box, return an xarray Dataset
- `get_data()` — high-level wrapper: download + process for a full time range

In [ ]:
import sys
import os

sys.path.insert(0, os.path.dirname(os.path.abspath('')))

import mrms
import numpy as np
import matplotlib.pyplot as plt

DATA_DIR = '/netfiles/ciroh/qpeData'
DATE = '20251201_0000'

---
## 1. QPE Radar Only — CONUS

`QPE_RADAR_ONLY` is the default product. The cells below download a single hourly file for CONUS, inspect the resulting Dataset, and plot it.

In [ ]:
qpe_file = mrms.download_mrms(
    date=DATE,
    data_dir=DATA_DIR,
    domain='CONUS',
    product=mrms.QPE_RADAR_ONLY
)
print(f'Downloaded: {qpe_file}')

### Inspect the Dataset

In [ ]:
ds_qpe = mrms.process_mrms(qpe_file, bbox=mrms.CONUS_BBOX)
ds_qpe

In [ ]:
vals = ds_qpe['cref'].values
print(f'Shape:     {ds_qpe["cref"].shape}')
print(f'Lat range: {float(ds_qpe.latitude.min()):.3f} — {float(ds_qpe.latitude.max()):.3f} °N')
print(f'Lon range: {float(ds_qpe.longitude.min()):.3f} — {float(ds_qpe.longitude.max()):.3f} °E')
print(f'Min QPE:   {np.nanmin(vals):.3f} mm/hr')
print(f'Max QPE:   {np.nanmax(vals):.3f} mm/hr')
print(f'Mean QPE:  {np.nanmean(vals):.3f} mm/hr')

### Map

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
ds_qpe['cref'].plot(
    ax=ax,
    cmap='YlGnBu',
    vmin=0,
    vmax=50,
    cbar_kwargs={'label': '1-hr QPE (mm)'}
)
ax.set_title('MRMS QPE Radar Only — CONUS — 2025-12-01 00:00 UTC')
plt.tight_layout()
plt.show()

---
## 2. Composite Reflectivity — CONUS

Download the same hour using `COMPOSITE_REFLECTIVITY` to verify the product parameter. CREF is measured in dBZ.

In [ ]:
cref_file = mrms.download_mrms(
    date=DATE,
    data_dir=DATA_DIR,
    domain='CONUS',
    product=mrms.COMPOSITE_REFLECTIVITY
)
print(f'Downloaded: {cref_file}')

In [ ]:
ds_cref = mrms.process_mrms(cref_file, bbox=mrms.CONUS_BBOX)

fig, ax = plt.subplots(figsize=(14, 7))
ds_cref['cref'].plot(
    ax=ax,
    cmap='gist_ncar',
    vmin=0,
    vmax=75,
    cbar_kwargs={'label': 'Composite Reflectivity (dBZ)'}
)
ax.set_title('MRMS CREF — CONUS — 2025-12-01 00:00 UTC')
plt.tight_layout()
plt.show()

---
## 3. Alaska Domain

Download and plot QPE for the `ALASKA` domain. The bounding box is `mrms.ALASKA_BBOX` (lat 50–72°N, lon −180–−129°W).

In [ ]:
ak_file = mrms.download_mrms(
    date=DATE,
    data_dir=DATA_DIR,
    domain='ALASKA',
    product=mrms.QPE_RADAR_ONLY
)
print(f'Downloaded: {ak_file}')

In [ ]:
ds_ak = mrms.process_mrms(ak_file, bbox=mrms.ALASKA_BBOX)

fig, ax = plt.subplots(figsize=(12, 6))
ds_ak['cref'].plot(
    ax=ax,
    cmap='YlGnBu',
    vmin=0,
    vmax=50,
    cbar_kwargs={'label': '1-hr QPE (mm)'}
)
ax.set_title('MRMS QPE Radar Only — Alaska — 2025-12-01 00:00 UTC')
plt.tight_layout()
plt.show()

---
## 4. Hawaii Domain

Download and plot QPE for the `HAWAII` domain. The bounding box is `mrms.HAWAII_BBOX` (lat 18–23°N, lon −161–−154°W).

In [ ]:
hi_file = mrms.download_mrms(
    date=DATE,
    data_dir=DATA_DIR,
    domain='HAWAII',
    product=mrms.QPE_RADAR_ONLY
)
print(f'Downloaded: {hi_file}')

In [ ]:
ds_hi = mrms.process_mrms(hi_file, bbox=mrms.HAWAII_BBOX)

fig, ax = plt.subplots(figsize=(8, 6))
ds_hi['cref'].plot(
    ax=ax,
    cmap='YlGnBu',
    vmin=0,
    vmax=50,
    cbar_kwargs={'label': '1-hr QPE (mm)'}
)
ax.set_title('MRMS QPE Radar Only — Hawaii — 2025-12-01 00:00 UTC')
plt.tight_layout()
plt.show()

---
## 5. Multi-hour range — get_data()

`get_data()` iterates over hourly timesteps, downloads and processes each file, and concatenates along a `time` dimension. Both `domain` and `product` are forwarded to each `download_mrms()` call automatically.

In [ ]:
ds_range = mrms.get_data(
    start_datetime='20251201_0000',
    end_datetime='20251201_0200',
    domain='CONUS',
    product=mrms.QPE_RADAR_ONLY,
    data_dir=DATA_DIR   # files already downloaded above — will be skipped
)
print(f'Dimensions: {dict(ds_range.dims)}')
print(f'Timestamps: {ds_range.time.values}')
ds_range

In [ ]:
n_times = len(ds_range.time)
fig, axes = plt.subplots(1, n_times, figsize=(7 * n_times, 5), constrained_layout=True)

for ax, ts in zip(axes, ds_range.time.values):
    ds_range['cref'].sel(time=ts).plot(
        ax=ax,
        cmap='YlGnBu',
        vmin=0,
        vmax=50,
        add_colorbar=False
    )
    ax.set_title(str(ts)[:16])

fig.colorbar(
    axes[0].collections[0], ax=axes,
    label='1-hr QPE (mm)', shrink=0.8
)
plt.show()

---
## 6. Module constants reference

In [ ]:
print('Supported domains:')
print(' ', mrms.DOMAINS)

print('\nDomain bounding boxes:')
for domain, bbox in mrms.DOMAIN_BBOX.items():
    print(f'  {domain}: {bbox}')

print('\nProducts:')
print(f'  QPE_RADAR_ONLY         = {mrms.QPE_RADAR_ONLY!r}')
print(f'  COMPOSITE_REFLECTIVITY = {mrms.COMPOSITE_REFLECTIVITY!r}')

print('\nURL template:')
print(f'  {mrms.MRMS_URL_TEMPLATE}')